<a href="https://colab.research.google.com/github/ecloguehwang/2026-Seojinhyeop-Workshop/blob/main/2.%EC%8B%9C%EA%B7%B8%EB%8B%88%EC%B2%98%EC%97%91%EC%85%80%EC%9E%90%EB%A3%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#시그니처엑셀자료
#석차에 배경색 없앤 것 + 인덱스 행 틀고정

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tabulate import tabulate
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter



#matplotlib에서 한글구현
plt.rc('font', family='NanumBarunGothic')

#그래프 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False


## 데이터 부르기
#path = '/content/'
#df = pd.read_excel(f'{path}sumyeong_2026_habbul.xlsx')

df = pd.read_excel('sumyeong_2026_habbul.xlsx')


# ── 전형분류 값 정리 ──────────────────────────────────────────
df.loc[df['전형분류'] == '일반(수능위주)', '전형분류'] = '수능'
df.loc[df['전형분류'] == '학생부',        '전형분류'] = '교과'

# ── 최종 열 NA/NaN → '불' 처리 ───────────────────────────────
df['최종'] = df['최종'].fillna('불')


# ── 전형분류별 색상 정의 ──────────────────────────────────────
TYPE_COLORS = {
    '교과': 'FFFF00',
    '종합': '90EE90',
    '논술': '800080',
    '실기': 'FFA500',
    '수능': 'A9A9A9',
}
TYPE_FONT_COLORS = {
    '교과': '000000',
    '종합': '000000',
    '논술': 'FFFFFF',
    '실기': '000000',
    '수능': '000000',
}

# ── 대학명에서 '학교' 제거 + 대학_모집단위 합치기 ──────────────
df = df.copy()
df['대학'] = df['대학'].str.replace('학교', '', regex=False)
df['대학_모집단위'] = df['대학'] + '\n' + df['모집단위']


# ══════════════════════════════════════════════════════════════
# [수정 1단계 & 2단계]
# 학번이 같고 전과목이 다른 행 → 동일 학생으로 판단
# 수시 포함 행의 전과목을 해당 학생의 대표 전과목(정렬 기준)으로 사용
# ══════════════════════════════════════════════════════════════

# 학번별로 수시 행의 전과목 값을 추출해 '수시_전과목' 열로 매핑
susie_grade = (
    df[df['지원시기'].str.contains('수시', na=False)]
    .drop_duplicates('학번')
    .set_index('학번')['전과목']
)
df['수시_전과목'] = df['학번'].map(susie_grade)

# 수시 전과목이 없는 경우(수시 미지원) 원래 전과목 값으로 대체
df['수시_전과목'] = df['수시_전과목'].fillna(df['전과목'])

# 기존 '전과목' 컬럼을 수시 기준 전과목으로 교체 (정렬 및 구간 분류 기준)
df['전과목_원본'] = df['전과목'].copy()
df['전과목'] = df['수시_전과목']


# ── 합격/불합격 분리 (수시_전과목 기준 정렬) ──────────────────
df_pass = df[df['최종'] == '합'].sort_values(by='전과목', ascending=True)
df_fail = df[df['최종'] == '불'].sort_values(by='전과목', ascending=True)


# ── 전체 등수 계산 (수시_전과목 기준, 낮을수록 1등) ────────────
df_all = pd.concat([df_pass, df_fail])
rank_map = (
    df_all.drop_duplicates('전과목')
          .sort_values('전과목')
          .reset_index(drop=True)
          ['전과목']
          .reset_index()
          .rename(columns={'index': '등수'})
          .assign(등수=lambda x: x['등수'] + 1)
          .set_index('전과목')['등수']
)

# 구간 정의
bins   = [1.0, 3.0, 4.0, 5.0, 6.0, 7.0, 9.1]
labels = ['1.0~2.99', '3.0~3.99', '4.0~4.99', '5.0~5.99', '6.0~6.99', '7.0~9.0']


def make_wide(df_group):
    if df_group.empty:
        return pd.DataFrame(), pd.DataFrame()

    grouped_text = df_group.groupby('전과목')['대학_모집단위'].apply(list)
    grouped_type = df_group.groupby('전과목')['전형분류'].apply(list)
    max_len = max(grouped_text.apply(len))

    text_wide = pd.DataFrame(
        grouped_text.apply(lambda x: x + [None] * (max_len - len(x))).tolist(),
        index=grouped_text.index
    )
    type_wide = pd.DataFrame(
        grouped_type.apply(lambda x: x + [None] * (max_len - len(x))).tolist(),
        index=grouped_type.index
    )
    return text_wide, type_wide


def build_sheet_df(df_pass_grp, df_fail_grp):
    pass_text, pass_type = make_wide(df_pass_grp)
    fail_text, fail_type = make_wide(df_fail_grp)

    all_indices = sorted(set(
        (pass_text.index.tolist() if not pass_text.empty else []) +
        (fail_text.index.tolist() if not fail_text.empty else [])
    ))
    if not all_indices:
        return pd.DataFrame(), {}

    def reindex_rename(df_wide, prefix, indices):
        if df_wide.empty:
            return pd.DataFrame(index=indices)
        df_wide = df_wide.reindex(indices)
        df_wide.columns = [f'{prefix}{i+1}' for i in range(len(df_wide.columns))]
        return df_wide

    pass_text = reindex_rename(pass_text, '합격', all_indices)
    fail_text = reindex_rename(fail_text, '불합격', all_indices)
    pass_type = reindex_rename(pass_type, '합격', all_indices)
    fail_type = reindex_rename(fail_type, '불합격', all_indices)

    result = pd.concat([pass_text, fail_text], axis=1)
    result.index.name = '내신등급'
    result = result.reset_index()

    # ── 석차 열 추가 (전체 등수 매핑) ────────────────────────
    result.insert(0, '석차', result['내신등급'].map(rank_map))

    # ── 수능등급 열 추가 ──────────────────────────────────────
    df_all_grp = pd.concat([df_pass_grp, df_fail_grp])
    suneung_map = df_all_grp.drop_duplicates('전과목').set_index('전과목')['등급수능']
    result.insert(2, '수능등급', result['내신등급'].map(suneung_map))
    result['수능등급'] = result['수능등급'].fillna('無')

    type_combined = pd.concat([pass_type, fail_type], axis=1).reset_index(drop=True)
    type_map = {}
    for r_idx in range(len(type_combined)):
        for col in type_combined.columns:
            val = type_combined.at[r_idx, col]
            if val:
                type_map[(r_idx, col)] = val

    return result, type_map


# 결과 저장 경로
#result_path = '/content/'
#output_file = f'{result_path}2025_00고_시그니처엑셀자료.xlsx'

output_file = '2026학년도_00고_시그니처엑셀자료_test.xlsx'


# ── 인덱스 열 목록 (흰색 배경 처리 대상) ─────────────────────
# ✅ 수정: '석차' 추가 → 석차 열 배경색 흰색으로 통일
INDEX_COLS = {'석차', '등수', '내신등급', '수능등급'}

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for label, low, high in zip(labels, bins[:-1], bins[1:]):
        mask_p = (df_pass['전과목'] >= low) & (df_pass['전과목'] < high)
        mask_f = (df_fail['전과목'] >= low) & (df_fail['전과목'] < high)

        final_df, type_map = build_sheet_df(df_pass[mask_p], df_fail[mask_f])

        if final_df.empty:
            pd.DataFrame({'내신등급': ['해당 데이터 없음']}).to_excel(
                writer, sheet_name=label, index=False)
            continue

        LEGEND_ROW = 1
        HEADER_ROW = 3
        DATA_START  = 4

        final_df.to_excel(writer, sheet_name=label, index=False, startrow=HEADER_ROW - 1)
        ws = writer.sheets[label]

        # ✅ 추가: 틀 고정
        # - 행: HEADER_ROW(3행)까지 고정 → DATA_START(4행)부터 상하 스크롤
        # - 열: 석차(1열) · 내신등급(2열) · 수능등급(3열) 고정 → 4열부터 좌우 스크롤
        ws.freeze_panes = ws.cell(row=DATA_START, column=4)

        thin   = Side(style='thin')
        thick  = Side(style='medium')
        border = Border(left=thin, right=thin, top=thin, bottom=thin)

        pass_cols = [c for c in final_df.columns if c.startswith('합격')]
        fail_cols = [c for c in final_df.columns if c.startswith('불합격')]
        col_names  = list(final_df.columns)

        # ── 합격/불합격 경계 열 인덱스 ───────────────────────
        boundary_col = (col_names.index(fail_cols[0]) + 1) if fail_cols else None

        # ── 범례 행 (맨 왼쪽) ──────────────────────────────
        legend_start_col = 1
        for i, ltype in enumerate(TYPE_COLORS.keys()):
            cell = ws.cell(row=LEGEND_ROW, column=legend_start_col + i)
            cell.value     = ltype
            cell.font      = Font(name='Arial', size=10, bold=True,
                                  color=TYPE_FONT_COLORS[ltype])
            cell.fill      = PatternFill('solid', start_color=TYPE_COLORS[ltype])
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border    = border
            ws.column_dimensions[get_column_letter(legend_start_col + i)].width = 8
        ws.row_dimensions[LEGEND_ROW].height = 20

        # ── 헤더 행 서식 ──────────────────────────────────────
        for col_name in INDEX_COLS:
            if col_name in col_names:
                col_idx = col_names.index(col_name) + 1
                cell = ws.cell(row=HEADER_ROW, column=col_idx)
                cell.font      = Font(name='Arial', size=10, bold=True, color='000000')
                cell.alignment = Alignment(horizontal='center', vertical='center')
                cell.fill      = PatternFill(fill_type=None)
                cell.border    = border

        if pass_cols:
            p_start = col_names.index(pass_cols[0]) + 1
            p_end   = col_names.index(pass_cols[-1]) + 1
            ws.merge_cells(start_row=HEADER_ROW, start_column=p_start,
                           end_row=HEADER_ROW,   end_column=p_end)
            mc = ws.cell(row=HEADER_ROW, column=p_start)
            mc.value     = '합격'
            mc.font      = Font(name='Arial', size=10, bold=True, color='000000')
            mc.alignment = Alignment(horizontal='center', vertical='center')
            mc.fill      = PatternFill(fill_type=None)
            for c in range(p_start, p_end + 1):
                if c == p_end and boundary_col:
                    ws.cell(row=HEADER_ROW, column=c).border = Border(
                        left=thin, right=thick, top=thin, bottom=thin)
                else:
                    ws.cell(row=HEADER_ROW, column=c).border = border

        if fail_cols:
            f_start = col_names.index(fail_cols[0]) + 1
            f_end   = col_names.index(fail_cols[-1]) + 1
            ws.merge_cells(start_row=HEADER_ROW, start_column=f_start,
                           end_row=HEADER_ROW,   end_column=f_end)
            mc = ws.cell(row=HEADER_ROW, column=f_start)
            mc.value     = '불합격'
            mc.font      = Font(name='Arial', size=10, bold=True, color='000000')
            mc.alignment = Alignment(horizontal='center', vertical='center')
            mc.fill      = PatternFill(fill_type=None)
            for c in range(f_start, f_end + 1):
                if c == f_start and boundary_col:
                    ws.cell(row=HEADER_ROW, column=c).border = Border(
                        left=thick, right=thin, top=thin, bottom=thin)
                else:
                    ws.cell(row=HEADER_ROW, column=c).border = border

        # ── 데이터 행 서식 ────────────────────────────────────
        for r_idx, row in enumerate(ws.iter_rows(min_row=DATA_START,
                                                  max_row=ws.max_row)):
            for cell in row:
                # ── 합격/불합격 경계 세로선 굵게 ────────────
                if boundary_col and cell.column == boundary_col:
                    cell.border = Border(
                        left=thick, right=thin, top=thin, bottom=thin)
                elif boundary_col and cell.column == boundary_col - 1:
                    cell.border = Border(
                        left=thin, right=thick, top=thin, bottom=thin)
                else:
                    cell.border = border

                cell.alignment = Alignment(wrap_text=True, horizontal='center',
                                           vertical='center')
                col_name = col_names[cell.column - 1] if cell.column <= len(col_names) else None

                if col_name in INDEX_COLS:
                    # ✅ 수정: 석차 포함 INDEX_COLS 열은 모두 흰색 배경
                    cell.font = Font(name='Arial', size=10)
                    cell.fill = PatternFill(fill_type=None)

                elif col_name and cell.value:
                    entry_type = type_map.get((r_idx, col_name))
                    if entry_type and entry_type in TYPE_COLORS:
                        bg = TYPE_COLORS[entry_type]
                        fc = TYPE_FONT_COLORS[entry_type]
                    elif col_name.startswith('합격'):
                        bg, fc = 'FFFACD', '000000'
                    else:
                        bg, fc = 'FFE4E1', '000000'
                    cell.fill = PatternFill('solid', start_color=bg)
                    cell.font = Font(name='Arial', size=10, color=fc)
                else:
                    cell.font = Font(name='Arial', size=10)
                    cell.fill = PatternFill(fill_type=None)

        # ── 열 너비 / 행 높이 ────────────────────────────────
        for col_idx in range(1, len(col_names) + 1):
            ws.column_dimensions[get_column_letter(col_idx)].width = 11
        for row_idx in range(DATA_START, ws.max_row + 1):
            ws.row_dimensions[row_idx].height = 38

        print(f'\n=== 내신등급 구간: {label} ===')
        print(tabulate(final_df, headers='keys', tablefmt='plain', showindex=False))

print(f'\n✅ 저장 완료: {output_file}')


=== 내신등급 구간: 1.0~2.99 ===
  석차    내신등급  수능등급    합격1                                                                     합격2                                                                     합격3                 합격4                 불합격1                                                                   불합격2                            불합격3                         불합격4                                 불합격5                            불합격6                 불합격7                       불합격8                    불합격9
     1       1.184  1.5         고신대                                                                    대구가톨릭대                                                              연세대                영남대                경상국립대                                                                부산대(양산)
                                의예과                                                                    의예과                                                                    전기전자공학부        의예과           